# CS 431: Data-Intensive Distributed Computing (Fall 2026)
# Assignment 3: Batch Analytics and Data Science

In Assignment 2, you worked with a dataset from the [Steam platform](https://store.steampowered.com/), built from the Steam API.
In this assignment, you'll build on the previous assignment to perform batch analytics with Spark and engage in typical data science tasks.
We'll provide you with the gold data warehouse starting point.

Imagine that you're still on the Steam data team.
The product team wants to understand some aspects of games on the platform and has asked you to help.

You're expected to use a coding agent to help you with this assignment.
However, you are ultimately responsible for and accountable for _everything_, including design decisions, quality checks, bugs, edge cases, interpretation of the results, etc.
You are also responsible for _understanding_ what the coding agent is doing.

**Start with a fresh database restore.**

This is to avoid possible confusion and leftover state from the previous assignment.

+ Drop and recreate the database `steam_2025` under the user `cs431`.
+ Restore `steam_gold_a3.dump` in the database, from `https://cs.uwaterloo.ca/~jimmylin/cs431-2026f/steam_gold_a3.dump` with MD5 checksum `2821b99dccbcec9ff927e6299726299a`.

Make sure you're not accidentally using tables in the `public` schema.
Everything below must be computed from tables in the `gold` schema in the above dump.

Let's start by setting up your computing environment.
Run the following cell, which should initialize Spark, the same as in the previous assignments.

If you get an error, stop and debug.

In [ ]:
import findspark, os

# TODO: Set this path for your machine.
os.environ["SPARK_HOME"] = "/Users/jimmylin/workspace/teaching2026/spark-4.2.0-bin-hadoop3"
findspark.init()

Start a local Spark session. 
We set shuffle partitions to eight (arbitrarily).

Again, just run the cell.
If you get an error, stop and debug.
You may get some warning messages, but it's safe to ignore them.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Assignment3")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.7")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

Use the following settings for PostgreSQL.

Keep the defaults in your final submission.
If you temporarily use another port or password, change the values back before submitting.

Just run the cell.
If you get an error, stop and debug.

In [ ]:
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))
POSTGRES_DB = os.getenv("POSTGRES_DB", "steam_2025")
POSTGRES_USER = os.getenv("POSTGRES_USER", "cs431")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

JDBC_URL = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
JDBC_PROPERTIES = {
    "user": POSTGRES_USER,
    "password": POSTGRES_PASSWORD,
    "driver": "org.postgresql.Driver",
}

def read_table(table_name):
    # Make sure it's a "gold" table:
    if not table_name.replace("_", "").isalnum():
        raise ValueError("Unexpected table name")

    return spark.read.jdbc(JDBC_URL, f"gold.{table_name}", properties=JDBC_PROPERTIES)

The above cell defines a function `read_table` that allows you to read an SQL table as a DataFrame.

Run the cell below and confirm that Spark can indeed read the database.
The following should print `150274`.
Don't continue until it does.

In [ ]:
print(read_table("fact_game_marketplace").count())

## 1. Understanding the "Gold" Tables

We'll start by loading the six gold tables needed in this assignment into DataFrames. 
For convenience, let's populate the `gold_tables` dictionary as follows:

+ `fact_games` → `gold.fact_game_marketplace`
+ `reviews` → `gold.reviews`
+ `bridge_game_genres` → `gold.bridge_game_genre`
+ `dim_genres` → `gold.dim_genre`
+ `bridge_game_platforms` → `gold.bridge_game_platform`
+ `dim_platforms` → `gold.dim_platform`

Run the following cell.

In [ ]:
gold_tables = {
    "fact_games": read_table("fact_game_marketplace"),
    "reviews": read_table("reviews"),
    "bridge_game_genres": read_table("bridge_game_genre"),
    "dim_genres": read_table("dim_genre"),
    "bridge_game_platforms": read_table("bridge_game_platform"),
    "dim_platforms": read_table("dim_platform"),
}

fact_games = gold_tables["fact_games"]
reviews = gold_tables["reviews"]
bridge_game_genres = gold_tables["bridge_game_genres"]
dim_genres = gold_tables["dim_genres"]
bridge_game_platforms = gold_tables["bridge_game_platforms"]
dim_platforms = gold_tables["dim_platforms"]

We have, in fact, created aliases for the DataFrames that correspond to the tables in the database.
Confirm that this is indeed so by running the following cell.

In [ ]:
if read_table("fact_game_marketplace").count() == gold_tables["fact_games"].count():
    print(True)
else:
    print(False)

if gold_tables["fact_games"].count() == fact_games.count():
    print(True)
else:
    print(False)

The output should be "True", twice.

As is typical before starting any data science task, it's important to get an understanding of "what's there".

Let's audit the data.
Construct `gold_table_audit`, a Spark DataFrame, with one row per entry in `gold_tables` and exactly these columns:

+ `table_name`: the key used in `gold_tables`
+ `row_count`: the total number of rows in that DataFrame
+ `column_count`: the number of columns in that DataFrame

Order by `table_name`.

In [ ]:
## acell_31_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(gold_table_audit.orderBy("table_name").toPandas().to_string(index=False))

Your row counts should be:

```text
bridge_game_genres      414269
bridge_game_platforms   194094
dim_genres                  77
dim_platforms                3
fact_games              150274
reviews                 865757
```

If they aren't, stop and debug. 

## 2. Building a Reusable Games DataFrame

Before answering product questions, we'll build a canonical DataFrame named `games`. 
This is the most important modeling step in the assignment.

### 2.1 Aggregate Review Metrics

Group `reviews` by `appid` to create the `review_stats` DataFrame with exactly these columns:

+ `appid`: the game identifier and grouping key
+ `derived_review_count`: number of review rows for the game
+ `derived_positive_review_count`: number of review rows where `voted_up IS TRUE`
+ `derived_negative_review_count`: number of review rows where `voted_up IS FALSE`
+ `positive_review_rate`: `derived_positive_review_count / derived_review_count`, as a numeric value between 0 and 1
+ `derived_average_author_playtime`: mean `author_playtime_forever` across the game's reviews; let Spark ignore null values in the mean


In [ ]:
## acell_32_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

review_stats

Running the above cell should produce something like `DataFrame[appid: ... ]`, which is the type of the variable.

Next, create a one-row `review_stats_audit` DataFrame with exactly these columns:

+ `reviewed_game_n`: number of rows in `review_stats`
+ `review_n`: sum of `derived_review_count`
+ `positive_review_n`: sum of `derived_positive_review_count`
+ `negative_review_n`: sum of `derived_negative_review_count`
+ `min_reviews_per_game`: minimum `derived_review_count`
+ `max_reviews_per_game`: maximum `derived_review_count`


In [ ]:
## acell_32_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(review_stats_audit.toPandas().to_string(index=False))

Sanity check these numbers.
Convince yourself that they are correct.

### 2.2 Join Review Metrics to Every Game

Left join `review_stats` to `fact_games` on `appid` and call the result `games`:

+ Retain every fact row and every original fact column.
+ Fill the three derived count columns with zero for games without sampled reviews.
+ Leave `positive_review_rate` and `derived_average_author_playtime` null when a game has no sampled reviews.
+ Do not overwrite or rename the stored zero-filled review columns in the facts table.
+ The result must contain every original fact column plus `derived_review_count`, `derived_positive_review_count`, `derived_negative_review_count`, `positive_review_rate`, and `derived_average_author_playtime`.


In [ ]:
## acell_32_3 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

games

Running the above cell should produce something like `DataFrame[appid: ... ]`, which is the type of the variable.

Let's audit what we've built.

Below, we create `games_audit`, a one-row DataFrame with:

+ `game_n`: total rows in `games`
+ `distinct_appid_n`: number of distinct `appid` values in `games`
+ `reviewed_game_n`: number of games where `derived_review_count > 0`
+ `unreviewed_game_n`: number of games where `derived_review_count = 0`
+ `review_n`: sum of `derived_review_count` across all games

(We're using the derived measures, not the stored zeroes.)

This audit is provided for you; just run the cell below.

In [ ]:
games_audit = games.agg(
    F.count("*").alias("game_n"),
    F.countDistinct("appid").alias("distinct_appid_n"),
    F.count(F.when(F.col("derived_review_count") > 0, 1)).alias("reviewed_game_n"),
    F.count(F.when(F.col("derived_review_count") == 0, 1)).alias("unreviewed_game_n"),
    F.sum("derived_review_count").alias("review_n"),
)

print(games_audit.toPandas().to_string(index=False))

Sanity check these numbers.
Convince yourself that they are correct.

**Q2.1: Explain why the join above must be a left join from `fact_games`, not an inner join.**

// acell_32_4 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

Create a DataFrame named `game_invariant_violations` that contains any `games` row that violates at least one of these rules:

+ `derived_review_count = derived_positive_review_count + derived_negative_review_count`
+ `positive_review_rate` is between 0 and 1 for reviewed games
+ `positive_review_rate` is null for unreviewed games

Select exactly `appid`, `derived_review_count`, `derived_positive_review_count`, `derived_negative_review_count`, `positive_review_rate`, and `derived_average_author_playtime`. Order by `appid`.


In [ ]:
## acell_32_5 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(f"invariant_violation_n={game_invariant_violations.count()}")

What do you get?

_(Nothing to actually write here.)_

### 2.3 Inspect the Sampling Cap

Create a DataFrame named `review_count_distribution` with one row per observed `derived_review_count` and exactly these columns:

+ `review_count`: the observed value of `derived_review_count`
+ `game_count`: number of rows in `games` with that value

Order by `review_count` ascending and include unreviewed games.
The cell below prints a small, fixed subset to make mistakes easy to spot; we will inspect the complete DataFrame when grading.

In [ ]:
## acell_32_6 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

check_counts = [0, 1, 2, 3, 4, 5, 10, 20, 50, 99, 100]
print(
    review_count_distribution
    .filter(F.col("review_count").isin(check_counts))
    .orderBy("review_count")
    .toPandas()
    .to_string(index=False)
)

**Q2.2: What does the spike at exactly 100 suggest about the data collection process?**

Why would it be misleading to treat `derived_review_count` as each game's lifetime review total?

// acell_32_7 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

Cache `games` and count the DataFrame by running the following cell:

In [ ]:
games = games.cache()
print(f"cached_game_n={games.count()}")

There should be 150274 games.

## 3. Statistical Batch Analysis

In this section, use Spark DataFrame transformations for all processing (joins, aggregations, etc.). 
You can convert small final result tables into Pandas for plotting, but only at the end.
Unless a question says otherwise, the unit of analysis is the game.

### 3.1 Which release cohorts are represented by the sampled reviews?


Restrict this analysis to release years 2015 through 2024, inclusive.
Create a DataFrame named `cohort_coverage` with exactly the following:

+ `release_year`: the cohort year and grouping key
+ `game_n`: number of games released in that year
+ `reviewed_game_n`: number of those games where `derived_review_count > 0`
+ `review_coverage_rate`: `reviewed_game_n / game_n`
+ `median_review_count`: median `derived_review_count` among reviewed games in the cohort; use `percentile_approx(..., 0.5, 10000)`
+ `proportion_at_100_among_reviewed`: number of games where `derived_review_count = 100`, divided by `reviewed_game_n`

Round rates to four decimal places.
Order by `release_year`.

In [ ]:
## acell_33_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(cohort_coverage.orderBy("release_year").toPandas().to_string(index=False))

Plot `review_coverage_rate` by `release_year` as a line graph below.
Make sure you:

+ Label both axes (what should the appropriate ranges be?)
+ Add a descriptive title
+ Include a horizontal grid to make the plot readable.

You may convert `cohort_coverage` to pandas if needed.

In [ ]:
## acell_33_2 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

import matplotlib.pyplot as plt

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

**Q3.1: Interpret the cohort table.**

The graph appears to show that review coverage is decreasing over time.
Is this actually true?

// acell_33_3 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

### 3.2 Which genres have the most positive sampled reviews?

Let's focus on games where `derived_review_count >= 20` and `positive_review_rate` is non-null.
Create this DataFrame, call it `eligible_reviewed_games`, and keep only:

+ `appid`: game identifier
+ `derived_review_count`: total sampled reviews for the game
+ `derived_positive_review_count`: positive sampled reviews for the game
+ `positive_review_rate`: game-level positive sampled reviews divided by total sampled reviews

The cell below does this for you: just run it.

In [ ]:
eligible_reviewed_games = (
    games
    .filter(
        (F.col("derived_review_count") >= 20)
        & F.col("positive_review_rate").isNotNull()
    )
    .select(
        "appid",
        "derived_review_count",
        "derived_positive_review_count",
        "positive_review_rate",
    )
)

print(f"eligible_game_n={eligible_reviewed_games.count()}")

Create a DataFrame named `genre_positivity` with exactly:

+ `genre_id`: genre identifier and grouping key
+ `genre_name`: the matching `name` from `dim_genres`
+ `reviewed_game_n`: number of distinct eligible `appid` values assigned to the genre
+ `sampled_review_n`: sum of `derived_review_count` across those games
+ `pooled_positive_rate`: sum of `derived_positive_review_count` divided by sum of `derived_review_count`
+ `mean_game_positive_rate`: unweighted mean of `positive_review_rate` across eligible games in the genre
+ `rate_gap`: unrounded `pooled_positive_rate` minus unrounded `mean_game_positive_rate`

Round all three rate columns to four decimals. 
Order by `pooled_positive_rate` descending, then `genre_id` ascending.

Only consider "eligible genres".
A genre is eligible only if it has at least 100 (`>=`) eligible games.

_Hint:_ You'll start with `eligible_reviewed_games` but will need to join in other tables...

In [ ]:
## acell_33_4 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(genre_positivity.toPandas().to_string(index=False))

Make a grouped horizontal bar chart for the top ten genres, with one bar for `pooled_positive_rate` and one for `mean_game_positive_rate`. 
Keep the table's ranking order, label the _x_-axis, include a legend, and use a 0-to-1 _x_-axis.

In [ ]:
## acell_33_5 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

**Q3.2: Why are the two rates different for each genre?**

Explain what a "typical review" and a "typical game" contribute to the two estimates.
Then state which you would use for the question “What fraction of sampled reviews in this genre are positive?”


// acell_33_6 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

### 3.3 Is price associated with game-level review positivity?

To answer this question, start from the `eligible_reviewed_games` DataFrame.

Let's analyze the games in terms of costs.
Define exactly five mutually exclusive groups, in this order:

1. `Free`: `is_free_game = true`
2. `$0.01-9.99`: `is_free_game = false` and `0.01 <= price_usd <= 9.99`
3. `$10.00-19.99`: `is_free_game = false` and `10.00 <= price_usd <= 19.99`
4. `$20.00-39.99`: `is_free_game = false` and `20.00 <= price_usd <= 39.99`
5. `$40.00+`: `is_free_game = false` and `price_usd >= 40.00`

Leave every other eligible game unassigned, including a paid game with a null, negative, or zero price.
Do not infer a price group.

First, sanity check: create a one-row `price_assignment_audit` DataFrame with:

+ `eligible_game_n`: total rows in `eligible_reviewed_games`
+ `assigned_game_n`: eligible games assigned to one of the five price groups
+ `excluded_game_n`: eligible games not assigned to any price group
+ `paid_null_price_n`: eligible games where `is_free_game = false` and `price_usd` is null

This makes the exclusion explicit before we analyze the assigned games.

In [ ]:
## acell_33_7 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(price_assignment_audit.toPandas().to_string(index=False))

You should have gotten the following:

```
eligible_game_n  assigned_game_n  excluded_game_n  paid_null_price_n
           10926            10348              578                578
```

If not, debug.

Next, create a DataFrame named `price_positivity` with one row per assigned price group and exactly the following:

+ `price_group`: one of the five labels defined above
+ `sort_order`: integers 1 through 5 in the order shown above
+ `game_n`: number of assigned eligible games in the group
+ `sampled_review_n`: sum of `derived_review_count` across those games
+ `mean_game_positive_rate`: unweighted mean of `positive_review_rate` across those games
+ `pooled_positive_rate`: sum of `derived_positive_review_count` divided by sum of `derived_review_count`
+ `median_game_positive_rate`: median `positive_review_rate`, using `percentile_approx(..., 0.5, 10000)`

Round rates to four decimals and order by `sort_order`.

In [ ]:
## acell_33_8 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(price_positivity.orderBy("sort_order").toPandas().to_string(index=False))

Now quantify uncertainty for the **mean game-level** rate. Create `price_intervals` with:

+ `price_group`: the same five labels as `price_positivity`
+ `sort_order`: the same integers 1 through 5
+ `game_n`: number of assigned eligible games in the group
+ `mean_game_positive_rate`: unweighted mean of game-level `positive_review_rate`
+ `standard_error`: sample standard deviation of game-level `positive_review_rate`, divided by `sqrt(game_n)`
+ `ci_low`: unrounded mean minus `1.96 × standard_error`
+ `ci_high`: unrounded mean plus `1.96 × standard_error`

Round the mean, standard error, and interval bounds to four decimals only after computing them. Order by `sort_order`.

In [ ]:
## acell_33_9 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(price_intervals.orderBy("sort_order").toPandas().to_string(index=False))

Plot the five means with vertical 95% confidence-interval error bars. Preserve price order, label both axes, use a 0-to-1 y-axis, and include a descriptive title.

In [ ]:
## acell_33_0 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

**Q3.3: Interpret the price analysis.**

Describe the observed association and identify the smallest group.
Can we conclude that a higher or lower price _causes_ differences in positivity ratings?
If so, explain why.
If not, provide two plausible confounders or selection effects.

// acell_33_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

### 3.4 How has cross-platform support changed by release cohort?

To answer this question we need to first reconstruct Windows, macOS, and Linux support.

Create a DataFrame called `reconstructed_platforms` with exactly:

+ `appid`: game identifier, with one row for every game in `fact_games`
+ `bridge_supports_windows`: `True` when the game has a bridge row whose dimension `name` is `windows`; otherwise `False`
+ `bridge_supports_mac`: `True` when the game has a bridge row whose dimension `name` is `mac`; otherwise `False`
+ `bridge_supports_linux`: `True` when the game has a bridge row whose dimension `name` is `linux`; otherwise `False`


In [ ]:
## acell_33_b (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

reconstructed_platforms

Confirm that `reconstructed_platforms` above has the requested fields.

For games released from 2010 through 2024 inclusive, compare each reconstructed flag with its counterpart in the facts table.

To do this, let's create a one-row DataFrame `platform_flag_audit` with:

+ `cohort_game_n`: number of games in the 2010–2024 population
+ `windows_mismatch_n`: games where `bridge_supports_windows` differs from `supports_windows`
+ `mac_mismatch_n`: games where `bridge_supports_mac` differs from `supports_mac`
+ `linux_mismatch_n`: games where `bridge_supports_linux` differs from `supports_linux`


In [ ]:
## acell_33_c (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(platform_flag_audit.toPandas().to_string(index=False))

The above is a sanity check... there shouldn't be any mismatches.

Next, based on `fact_games`, create a DataFrame named `platform_cohorts` for 2010–2024 with:

+ `release_year`: the cohort year and grouping key
+ `game_n`: number of games released in that year
+ `windows_rate`: proportion of cohort games where `supports_windows = true`
+ `mac_rate`: proportion of cohort games where `supports_mac = true`
+ `linux_rate`: proportion of cohort games where `supports_linux = true`
+ `mean_platform_count`: cast each of the three Boolean support flags to an integer, sum them per game, then take the cohort mean

Round the three rates and `mean_platform_count` to four decimals.
Order by `release_year`.

In [ ]:
## acell_33_d (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(platform_cohorts.orderBy("release_year").toPandas().to_string(index=False))

Plot Windows, macOS, and Linux rates as three lines on the same axes. Use points, a legend, a 0-to-1 y-axis, axis labels, a descriptive title, and a light grid.


In [ ]:
## acell_33_e (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########


**Q3.4: Interpret the platform trends.**

Describe the most obvious trend.
One hypothesis is that individual developers have become less interested in cross-platform support over the last decade or so.
Is this hypothesis supported by the data?

// acell_33_f (keep this id for tracking purposes)

**Answer:** Replace with your answer.

## 4. Distributed Query Optimization

The dataset we've given you fits on one machine... so you'll have to think through what happens if your dataset is 10, 100, 1000 times larger...


Below, we're giving you two DataFrames: `fact_unpartitioned` and `fact_partitioned`.

(Just run the cell.)

In [ ]:
bounds = fact_games.agg(F.min("appid").alias("lower"), F.max("appid").alias("upper")).first()
selected_columns = ["appid", "release_year", "price_usd", "is_free_game"]

fact_unpartitioned = (
    spark.read.jdbc(JDBC_URL, "gold.fact_game_marketplace", properties=JDBC_PROPERTIES)
    .select(*selected_columns)
)

fact_partitioned = (
    spark.read.jdbc(
        JDBC_URL,
        "gold.fact_game_marketplace",
        column="appid",
        lowerBound=int(bounds["lower"]),
        upperBound=int(bounds["upper"]),
        numPartitions=8,
        properties=JDBC_PROPERTIES,
    )
    .select(*selected_columns)
)

**Q4.1: Explain differences in the two DataFrames.**

Explain how the two DataFrames are the same or different.

// acell_34_1 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

**Q4.2: Execution Implications.**

What are the implications in terms of analytical queries starting from one or the other DataFrame?
For example, what do you think might happen if you run a query on a cluster with one machine?
Two machines?
Ten machines?
One hundred machines?

// acell_34_2 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

Unsurprisingly, the first thing we're going to do is audit the two DataFrames above.

Create a DataFrame named `jdbc_read_audit` with the following columns:

+ `read_name`: `partitioned` or `unpartitioned`
+ `row_count`: total rows returned by that read
+ `spark_partition_count`: `DataFrame.rdd.getNumPartitions()` for that read
+ `min_appid`: minimum `appid` returned by that read
+ `max_appid`: maximum `appid` returned by that read

Order by `read_name`.
The DataFrame should have exactly two rows.

_Hint:_ you'll need RDDs for this.

In [ ]:
## acell_34_3 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(jdbc_read_audit.orderBy("read_name").toPandas().to_string(index=False))

Let's examine `fact_partitioned` in more detail and count the number of records in each partition.

Create a DataFrame called `jdbc_partition_sizes` with exactly:

+ `partition_id`: (zero-based) partition index
+ `row_count`: number of rows in that partition

Order by `partition_id` and print the result.

_Hint:_ you'll need RDDs for this.

In [ ]:
## acell_34_4 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(jdbc_partition_sizes.orderBy("partition_id").toPandas().to_string(index=False))

**Q4.3: Partition Sizes.**

Does each partition contain the same number of rows?
Why or why not?

// acell_34_5 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

_Interesting._

Let's examine the impact of partitioning on execution times.

(Again, be careful here: you're running in local mode on a single machine.)

Let's measure the execution times of the following query:

In [ ]:
import time
import statistics

def time_yearly_summary(facts):
    start = time.perf_counter()

    benchmark_games = facts.join(review_stats, on="appid", how="left")

    yearly_summary = (
        benchmark_games
        .groupBy("release_year")
        .agg(
            F.count("*").alias("game_n"),
            F.count("derived_review_count").alias("reviewed_game_n"),
            F.sum(F.coalesce(F.col("derived_review_count"), F.lit(0))).alias("review_n"),
            F.avg("positive_review_rate").alias("mean_positive_review_rate"),
        )
    )

    yearly_summary.collect()

    return time.perf_counter() - start

We can run the query with either DataFrame:

```
elapsed = time_yearly_summary(fact_partitioned)
elapsed = time_yearly_summary(fact_unpartitioned)
```

Your task:
run each query ten times and throw away the first three trials (as warmup).
Compute the mean and variance over the next seven trials.
(So in total, you'll run 20 queries.)

Summarize the results in a DataFrame named `timing_results`, with exactly two rows and three columns:

+ `type`: `partitioned` or `unpartitioned`
+ `mean`: mean over the seven trials (in seconds)
+ `variance`: sample variance over the seven trials (in seconds squared)

In [ ]:
## acell_34_6 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

print(timing_results.orderBy("type").toPandas().to_string(index=False))

**Q4.4: Execution Times.**

Are the results what you expected?
Why or why not?
What do you think would happen if `gold.fact_game_marketplace` were one thousand times larger, on a cluster of 10 machines?

// acell_34_7 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

All done with Spark!
Stop the session.

In [ ]:
spark.stop()

## 5. Reflection on AI Collaboration

Reflect on your use of AI to complete this assignment.

Complete this log honestly.
This is primarily for your own benefit.
We'll read it, but it won't be graded.

Potential things to reflect on:

+ What I asked Codex to do.
+ What worked.
+ What failed or needed correction.
+ What I changed or checked myself.
+ One place where my domain knowledge or lecture knowledge mattered.
+ One prompt I would write differently next time.


// acell_35_1 (keep this id for tracking purposes)

**Answer:** Replace with your answer.

## 6. Assignment Submission


Before submitting:

+ Restart the kernel and run all cells from top to bottom against a fresh database restore.
+ Make sure every required table is computed from the `gold` schema.
+ Keep every `acell_...` identifier and every instructor-provided output cell unchanged.
+ Remove temporary debugging output, but keep all required tables, plans, plots, and written answers.

For submission details, see [the course homepage](https://lintool.github.io/cs431-2026f/assignments/assignment3.html).
Make sure you conform to the requirements articulated there.